# Reading the ball

**A traceable walkthrough of our frame-level possession model.**

More than a Hack 2027 · Barcelona vs Athletic Club. This page explains, in plain football terms, how we label every frame of one match with who has the ball. It needs no machine-learning background. The code behind every number is in `solution/`, and the appendix maps each number to its source. Our method is frozen. A late section re-measures some small parts of it and says how far to trust them.

**How to read it.** The box below is the five-minute version. Then comes the full story, from the problem to the limits. The glossary and the traceability appendix are at the end. Some figures are clips that play only in our local copy; this public copy shows a labelled placeholder instead.

| **Start here** | the whole story, in five lines |
|---|---|
| **The task** | One label per frame: **H** Barcelona, **A** Athletic, **D** dead ball (a stoppage), **X** not graded. |
| **The score** | **spellF1**, the ranking score, matches each true possession to a predicted one by time overlap (**IoU**: shared time over total time). It averages the F-score over five overlap levels, 0.3 to 0.7. A boundary that should not exist costs twice: a false positive, and a true possession cut in two. |
| **What we built** | **Baseline+**, our first stage: the nearest player has the ball, or else the last owner. A **gate** that calls a long stretch with no tracked ball and still players a stoppage. An **in-gap decoder** that reads team shape to pick the owner in a stretch the gate leaves live. Three later rules clean up (see the pipeline section). |
| **How much better** | On ten practice matches the gate adds +.0728 [+.0649, +.0808] spellF1 over Baseline+, and the full stack +.0988 [+.0909, +.1068]. Baseline+ is our own first stage, not the organizers' reference (`smooth50`): the gate-plus-decoder stack beats that reference by +.1003 [+.0877, +.1130], and the final build was never paired with it. The decoder alone adds +.0017, **level with zero** under our .004 rule. |
| **The main limit** | The practice matches are easier than ours. The decoder can rewrite only 3.9% of our graded frames. The five public minutes with answers hold 17 true possessions and are **report-only**: reported, never used to choose anything. |

## 1. The problem: a ball that keeps disappearing

Picture a passage from our match. Barcelona moves the ball in their own half, the players jog into shape, and the tracking loses the ball. Six seconds later it is still gone and the players have stopped. The answer strip under the pitch turns gray: the stretch is **dead**. The figure below is that passage as a clip.

<img src="figs/public/fig_match_gap.png" style="width:100%">
<div class="caption" style="font-size:0.9em; color:#555; margin:2px 0 14px 0;">Our match: live play, then the tracking loses the ball and the label turns D. Withheld in this public copy because it renders real tracking.</div>

The clip covers twenty seconds of our match, rendered from the submission's own tracking file. The gray band is a **ball-missing run**, frames with no ball position at all; the red band is the model's **D**. The ball is missing for 40.7% of graded frames, and almost all of that time is a stoppage. Everything else on this page follows from that fact.

The task is one of **three labels** per graded frame: **H** Barcelona, **A** Athletic, **D** dead ball. A **possession** is a maximal unbroken run of H or A; D ends it. We answer at 25 Hz for 158,857 of 199,319 frames, and the sequence must be coherent.

<img src="figs/fig_three_labels.png" style="width:100%">
<div class="caption" style="font-size:0.9em; color:#555; margin:2px 0 14px 0;">The three labels, and the 25 answers per second they ask for. Illustration, not match data.</div>

The obvious first rule is **who is nearest the ball**, the organizers' starter baseline, and with a 2.5 m radius it is a good start. But nearest is not **owner**: the owner is the team that last touched the ball and keeps it through a flight, a dribble or a scramble. In the schematic the defender is nearest, but the ball is still travelling from passer to receiver.

<img src="figs/fig_nearest_player.png" style="width:100%">
<div class="caption" style="font-size:0.9em; color:#555; margin:2px 0 14px 0;">Why the nearest player is not the owner: the ball is in flight, and the nearest player has not touched it. Illustration, not match data.</div>

On clean practice data nearest catches .764 of visible owner changes; degraded to our noise, .482, and about 13.5% cannot be resolved from geometry at all. Hence two other clues: **missingness** and **team shape**.

## 2. The score: why one extra boundary is expensive

One true possession: Barcelona has the ball for ten seconds, then the whistle goes. The animation is synthetic and scored by the organizer's own metric, not a cartoon. It explains almost every design choice we made.

<img src="figs/fig_spellf1.gif" style="width:100%">
<div class="caption" style="font-size:0.9em; color:#555; margin:2px 0 14px 0;">The centerpiece: a true ten-second possession, a perfect answer, the same answer split, and a missed spell, scored by the organizer's metric. Illustration, not match data.</div>

The score is **spellF1**. Possessions are matched one-to-one by **temporal IoU**, the shared frames divided by the union. The F-score is averaged over five overlap levels, 0.3 to 0.7. The right team is not enough if the interval is in the wrong place.

A perfect answer scores 1.00. Split it in two with a one-second D gap: .40, with each fragment at IoU 0.50. That is inside the range, but only one can match and the other is a false positive. Split in three: .10, IoU 0.33, at the edge of the range. Split in four: .00, every fragment at IoU 0.25, below the lowest level, 0.3, so nothing matches. Miss it completely: also .00.

<img src="figs/fig_spellf1_curve.png" style="width:100%">
<div class="caption" style="font-size:0.9em; color:#555; margin:2px 0 14px 0;">The same experiment as a curve: score against the number of fragments. Illustration, not match data.</div>

A second synthetic timeline makes the asymmetry concrete: the exact answer, a one-frame gap near the middle of a possession, and a short possession predicted dead. The gap scores below the miss, because it pays twice.

<img src="figs/fig_spellf1_explainer.png" style="width:100%">
<div class="caption" style="font-size:0.9em; color:#555; margin:2px 0 14px 0;">One synthetic timeline, three answers, scored by the organizer's metric: exact, a one-frame gap near the middle of a possession, and a missed short possession. The gap costs more than the miss.</div>

The asymmetry is the point. A missed possession costs one false negative. A boundary that should not exist costs a false positive *and* cuts a true possession in two, so that possession must now compete with its own fragment. Hence our caution about hand-overs, and why a clue can raise frame accuracy and still lose score. Differences below about 0.04 are noise, and the public five minutes hold only 17 possessions, so we never choose a method on them.

## 3. The practice set: ten matches, made to look like ours

We have the true answers for five public minutes, and the rules forbid tuning on them: 17 possessions cannot choose a threshold, and tuning there picks the wrong answer even for a one-parameter baseline. Every choice is made on other professional matches with public answers, after we make their data look like ours.

The next figures show one practice passage, clean and **degraded**. Clean, the ball stays visible through the stoppage, so a ball-based rule sees it for free. Degraded, the ball is dropped where our tracker drops it, the path is smoothed and the gap edges jittered: the stoppage is still there in the players, the ball is not.

<div style="display:flex;"><div style="flex:1; padding:0 4px;"><center><img src="figs/public/fig_practice_clean.png" style="width:100%">
<div class="caption" style="font-size:0.9em; color:#555; margin:2px 0 14px 0;">The practice clip before degrading: the ball stays visible through the stoppage. Withheld in this public copy because it renders real tracking.</div></center></div><div style="flex:1; padding:0 4px;"><center><img src="figs/public/fig_practice_degraded.png" style="width:100%">
<div class="caption" style="font-size:0.9em; color:#555; margin:2px 0 14px 0;">The same clip after degrading: the ball is missing where our tracker loses it. Withheld in this public copy because it renders real tracking.</div></center></div></div>

The practice set is **10 matches** from two public providers: seven German-league (DFL) and three Metrica. A third provider, SkillCorner, gives 20 more matches. They are used only for who-has-the-ball questions and never pooled. The ten are frozen: same games, same random seeds, same degrader, same six configurations.

<img src="figs/fig_missingness.png" style="width:100%">
<div class="caption" style="font-size:0.9em; color:#555; margin:2px 0 14px 0;">Where the ball is missing: our match against the degraded practice set. Aggregates, no match frames.</div>

The set had to be repaired. Our match hides the ball for 40.7% of graded frames, the degraded practice set for 37.3%, close enough. The share inside live play did not match: the old degrader dropped the ball at random, including in live play, so 42% of practice live frames had no ball against ~3% of ours. After the fix it is 4.4%, and 2.1% of long gaps are live against ≤ ~4% of ours. The old gap taught the model the opposite of the truth.

Two flaws, both fixed. The first: the **resting ball** shook about twenty times too much, so ball motion separated dead from live far too well. We measure that separation as an **AUC**, where lower is harder and a perfect split is one. Fixing the sign took ball-motion-only separation from perfect to .92, and .907 on twenty held-out SkillCorner games. Both are still above our .80 target, so ball-motion clues remain optimistic upper bounds. The second flaw was **where the ball hides**: after the fix Baseline+ moved from .216 to .326 and the in-gap gain fell from +.082 to +.006. The set is also about .06 easier than the match: .3037 against .243 for the two-second baseline. So we trust the ranking, never the level.

## 4. What is an in-gap decoder?

A **possession** is a run of frames in which one team has the ball; a dead ball ends it. The tracking loses the ball on 40.7% of our graded frames, usually in blocks of a few seconds. That is the central difficulty: for those frames the labels must say either *play continues, this team has it* or *the game is stopped*, and the ball itself cannot tell us which.

Take one invented passage, twelve seconds long. The ball is visible for four seconds, the tracker loses it for four, and it comes back. The players never stop moving. The gap could be a stoppage — a throw-in, an injury, a goal kick — or a tracking dropout while play continues. Two fixed rules both fail: call the whole gap dead and one possession becomes two; carry the last owner and the labels are wrong for the rest of the gap whenever the ball changed hands.

<img src="figs/fig_gap_ambiguity.png" style="width:100%">
<div class="caption" style="font-size:0.9em; color:#555; margin:2px 0 14px 0;">One passage: the ball is visible, missing for four seconds, visible again. Calling the whole gap dead splits one possession; carrying the last owner hides a hand-over. Illustration, not match data.</div>

The first error is the expensive one. A spurious dead stretch creates a boundary, so the score sees two short possessions where the truth has one. The second error costs team accuracy inside the gap but leaves the spell count alone. The gate is built for the first problem; the decoder is built for the second. The gate decides whether the gap is dead; the decoder decides who has the ball when it is not.

The **in-gap decoder** resolves the gap from the players, not the ball. It runs only inside a ball-missing stretch of at least two seconds, and only on frames the gate left live — the gate still owns the dead-ball call. Inside the gap it asks which team looks like it has the ball: the team that is spread out, with players ahead of the ball's last position, is usually the one in possession; the defending team is compact around the ball. It scores every frame for the home team, then smooths the sequence with a penalty for changing team, so one noisy frame cannot create a boundary.

<img src="figs/fig_ingap_steps.png" style="width:100%">
<div class="caption" style="font-size:0.9em; color:#555; margin:2px 0 14px 0;">The decoder in four steps: it works only inside a gap the gate left live, reads the players' shape, scores every frame for the home team, then smooths the sequence with a switching penalty. Illustration, not match data.</div>

In the toy frame the home team is spread across the pitch and the away team is compact around the ball's last position, so the ball-free probability of home possession, **P(H)**, sits near .8 across the gap. The switching penalty is what turns that noisy curve into one H label: without it, a single dip would write A and cut the spell.

The same passage under the fixed rule and under the decoder. If the gap is called dead, the two four-second spells each cover only a third of the true twelve-second possession: IoU .33, so at the lowest threshold one matches and the other is a false positive, and at the middle threshold and above neither matches. With the decoder the gap is H, one spell covers the possession at IoU 1.00.

<img src="figs/fig_ingap_before_after.png" style="width:100%">
<div class="caption" style="font-size:0.9em; color:#555; margin:2px 0 14px 0;">The same passage under the fixed rule (call the gap dead: two short spells) and with the decoder (one spell). Each split half overlaps the true possession at IoU .33; the continuous spell at 1.00.</div>

That is the argument for the stage: a handful of frames can decide whether a twelve-second possession is found or cut in two. It is also why the measured gain is small — the decoder can help only where a gap is both live and long enough, and every boundary it writes is a chance to be wrong.

**Where it sits.** The pipeline does two jobs and a clean-up. It decides the owner while the ball is visible, and dead ball while it is not. Then it removes the seams. The decoder re-opens the owner question inside a live gap. The gate, grown D (extend a visible resting ball to its stoppage) and the sub-gap rule (a second, weaker gate) decide dead ball. The short-spell floor (delete spells under a second and a half) cleans up.

<img src="figs/fig_ingap_locator.png" style="width:100%">
<div class="caption" style="font-size:0.9em; color:#555; margin:2px 0 14px 0;">Where the in-gap decoder sits: after the gate and grown D, before the sub-gap rule and the floor.</div>

**What it buys.** On the ten practice games, removing the decoder from the final stack changes spellF1 by +.0017 [+.0004, +.0030], with 3/10 games negative and 6/6 primary cells non-negative. Under our .004 rule that is **level with zero**: the sign is stable, the size is not established. It was credited +.0052 [+.0038, +.0065] when it landed on the gate-only stack (the pipeline before the later rules). On the full stack, grown D and the short-spell floor now do much of that work — two-thirds of the drop is those two stages, the rest is the sub-gap rule. Mechanically it adds 2,289 spells of a second and a half or more and removes 921 shorter ones, so flicker falls from 6.25% to 4.18%, with the dead share unchanged.

<img src="figs/fig_ingap_result.png" style="width:100%">
<div class="caption" style="font-size:0.9em; color:#555; margin:2px 0 14px 0;">The decoder's measured marginal on the practice set: +.0017, inside the .004 band, so level with zero; the gate-only +.0052 is what it was credited when it landed. Chart from measured numbers.</div>

In short, the idea is worth understanding, but the gain is not our headline. The decoder is kept because its sign is positive in every primary cell and the ball-free evidence is what makes hidden-ball stretches legible at all; on our match only 3.9% of graded frames are eligible, and with the gate in front it rewrites only a small fraction of them.

## 5. The pipeline, one stage at a time

The section above explains the in-gap decoder. This one takes the six stages in the order we met them and shows a passage for each. No single passage shows all six, because they need different conditions. The gate needs a long gap. Grown D needs a visible resting ball. The decoder needs a gap the gate left live. The sub-gap rule needs one the gate left alone. So the clips come from four passages. Stages one to three share one; each later clip shows where that stage changes a label.

**Stage 1 — nearest player, smoothed.** The organizers' two-second baseline: nearest player within a radius, majority-smoothed over fifty frames. The reference to beat: .3037.

<img src="figs/public/fig_stage_smooth50.png" style="width:100%">
<div class="caption" style="font-size:0.9em; color:#555; margin:2px 0 14px 0;">Stage 1, the organizers' 2-second baseline: nearest player, majority-smoothed. Withheld in this public copy because it renders real tracking.</div>

**Stage 2 — hysteresis and merge.** *Hysteresis* means a player must clearly take the ball before the owner changes. A player within 2.5 m owns the ball, otherwise the last owner keeps it; short spells are *merged* into their neighbors. It scores .3259, +.0222 [+.0058, +.0403] over that reference, but that lead does not survive a half-second shift, so read it as *about level to slightly ahead*. Too much hysteresis absorbs a real hand-over; too little emits flickers, which cost twice.

<img src="figs/public/fig_stage_baseline_plus.png" style="width:100%">
<div class="caption" style="font-size:0.9em; color:#555; margin:2px 0 14px 0;">Stage 2, Baseline+: hysteresis, merge and a 2.5 m radius. Withheld in this public copy because it renders real tracking.</div>

**Stage 3 — the missing-ball gate.** Ball gone for a while and players still: the stretch becomes D. The largest gain in the stack, +.0728 [+.0649, +.0808] over Baseline+, no game negative; expected +.06 to +.08 on our match. A wrong gate turns live possession into a stoppage and cuts a spell, the most expensive error the score charges.

<img src="figs/public/fig_stage_gate.png" style="width:100%">
<div class="caption" style="font-size:0.9em; color:#555; margin:2px 0 14px 0;">Stage 3, the missing-ball gate: the ball is gone and the players are still, so the stretch becomes D. Withheld in this public copy because it renders real tracking.</div>

**Stage 4 — the in-gap decoder.** In a missing-ball gap, team shape says which team is wider and more likely to have the ball, so the decoder may insert a hand-over. It adds +.0052 [+.0038, +.0065], but only 3.9% of our frames are eligible, so expected +.002 to +.005. The clip shows the mechanism, not a success: an H island where the answer stays A.

<img src="figs/public/fig_stage_ingap.png" style="width:100%">
<div class="caption" style="font-size:0.9em; color:#555; margin:2px 0 14px 0;">Stage 4, the in-gap decoder: team shape inserts a hand-over inside a missing-ball gap. Withheld in this public copy because it renders real tracking.</div>

**Stage 5 — grown stoppages.** A resting ball is a real stoppage but only a sliver. Extend each moment to the stoppage around it, with a radius and a thirty-second cap: +.0055 [.0044, .0066], expected +.001 to +.005. Only safe on top of the gate; on raw Baseline+ most changed frames are live play.

<img src="figs/public/fig_stage_grown.png" style="width:100%">
<div class="caption" style="font-size:0.9em; color:#555; margin:2px 0 14px 0;">Stage 5, grown stoppages: each resting-ball moment is extended to the stoppage around it. Withheld in this public copy because it renders real tracking.</div>

**Stage 6 — the sub-gap rule.** Some stoppages sit inside gaps the gate left alone. A weaker rule fires when the stoppage probability stays above one half for two seconds: +.0044 [.0033, .0056], expected about +.002. On a live gap it creates a D island inside a possession.

<img src="figs/public/fig_stage_subgap.png" style="width:100%">
<div class="caption" style="font-size:0.9em; color:#555; margin:2px 0 14px 0;">Stage 6, the sub-gap rule: a stoppage inside a gap the main gate left alone. Withheld in this public copy because it renders real tracking.</div>

The code runs them in a different order from the list above: gate, grown D, in-gap, sub-gap, then the short-spell floor. Grown D must be excluded from the re-decode, and the sub-gap rule must see the gate's output. The floor is not a seventh stage but a last clean-up, and it **is** in the final build. Deleting H/A spells under one and a half seconds adds +.0099 [.0063, .0138] on the gate-plus-decoder stack and +.0113 on the final labels. It is a trade: real short possessions are deleted to remove more false ones.

The gains in the table are measured one stage at a time, so they are not a running sum. The practice column is blank for grown D and the sub-gap rule because neither has a standalone score; each was measured only as a paired difference.

| stage | what it does | practice spellF1 | marginal gain | expected on our match |
|---|---|---|---|---|
| smooth50 | nearest player, majority-smoothed | .3037 | — | — |
| Baseline+ | hysteresis, merge, 2.5 m radius | .3259 | +.0222 [+.0058, +.0403] | level to slightly ahead |
| gate | missing ball + still players -> D | .3987 | +.0728 [+.0649, +.0808] | +.06 to +.08 |
| in-gap decoder | team shape inside a missing-ball gap | .4039 | +.0052 [+.0038, +.0065] | +.002 to +.005 |
| grown stoppages | resting ball -> whole stoppage | — | +.0055 [.0044, .0066] | +.001 to +.005 |
| sub-gap rule | second gate inside unfired gaps | — | +.0044 [.0033, .0056] | +.002 |

## 6. The two clues that carry it

In the clip from the first section the ball is gone for six seconds. Two independent checks were asked whether that time is a stoppage: a classifier trained on player motion, and the broadcast camera's own ball track, which follows the ball through gaps our tracking misses. Neither uses the answers.

<img src="figs/fig_gate_instruments.png" style="width:100%">
<div class="caption" style="font-size:0.9em; color:#555; margin:2px 0 14px 0;">Two independent instruments agree on how much of the missing time is dead. Aggregates, no match frames.</div>

The player-motion classifier flags 85.6% [80.8%, 89.2%] of our missing time as dead. The broadcast camera's track puts 84.5% of it dead. A further slice, 2.1%–6.9% of the time, is confirmed live, which carries the dead share to about 82%–87%. After correcting for what each check cannot see, the probability that a missing frame is dead is .93–.99. Two checks with different failure modes agreeing is the strongest label-free evidence we have. It is not a license to call every gap dead, so the gate also requires still players.

The second clue is **team shape**. The team with the ball spreads out; the defender stays compact. On live practice frames the possession team's width is 13.9 m against 11.3 m, a difference of about 2.5 m that separates them with an **AUC** (a score where a coin flip is one half and a perfect separator is one) of about .88–.90. The catch is where it applies. Inside a dead gap the same difference collapses to −0.4 m, because both teams walk into position. So it works only where the gap is live. Only 3.9% of our frames are eligible, against 17% to 44% on the practice set. Hence +.002 to +.005, not the large number the decoder scored alone.

<img src="figs/fig_width_schematic.png" style="width:100%">
<div class="caption" style="font-size:0.9em; color:#555; margin:2px 0 14px 0;">The team with the ball is wider; the defending team is narrow. Illustration, not match data.</div>

<img src="figs/public/fig_shape_deform.png" style="width:100%">
<div class="caption" style="font-size:0.9em; color:#555; margin:2px 0 14px 0;">The shape clue in a real missing-ball gap: the wide team changes as the owner changes. Withheld in this public copy because it renders real tracking.</div>

## 7. What we tried and dropped

The most tempting idea was to find the kick from how the ball moves. On clean public data the textbook detector finds 75%–83% of them. On our ball it fails, because the path is smoothed over about a third of a second. Smoothing the practice ball the same way reproduces the failure.

That set the pattern: a clue that works on clean data is not evidence until it survives our noise. The table gives the number that killed each idea. The chart compares four decoding ideas with our full build. Viterbi and MBR are two standard ways to turn per-frame probabilities into one label sequence (see the glossary).

The duration model was the expensive attempt. An explicit-duration **HSMM** (a sequence model that also learns how long possessions last) scores −.0090 [−.0163, −.0024], negative in nine of ten games. A sticky **HMM** (the same model without the duration prior) scores +.0084 [−.0025, +.0180]. That is positive but includes zero, so not yet trustworthy. The duration prior is what loses.

Four smaller ideas also failed. Guessing the restarting team from the type of stoppage changed accuracy by −0.4 points. Splitting long dead blocks scores −.0011. Crowd loudness led by +.027, below our five-point minimum. A public sound-event model heard a synthetic whistle at .115 but the real one at .001.

| idea | what killed it |
|---|---|
| kicks from ball kinematics | only 75%-83% recall on clean data; our ball is smoothed, so it fails here too |
| ball-player coupling | cost −.002 to −.011 spellF1 when added to the decoder |
| video ball detection | 36% recall on frames where tracking had the ball, 82% false positives on frames where it did not; the ball is 8 px to 10 px |
| explicit-duration HSMM | −.0090 [−.0163, −.0024], negative in nine of ten games |
| restart team by stoppage type | −0.4 points restart accuracy and negative spellF1 in every variant |
| split long stoppage blocks | −.0011, negative in ten of ten games |
| crowd loudness | lead +.027, below the five-point floor |
| public audio-event model | synthetic whistle .115, this match's whistle .001 |

<img src="figs/fig_killed.png" style="width:100%">
<div class="caption" style="font-size:0.9em; color:#555; margin:2px 0 14px 0;">Each dropped decoding idea's spellF1 change against the shipped build, on the practice set. Every bar points left: each idea lost. Drawn from measured numbers.</div>

## 8. Results on the final build

The method is frozen. These are the practice-set results: ten practice games, six primary cells, game-cluster intervals. The final-build row is **one run of the whole stack**, not a sum of the component deltas above.

| row | practice-set spellF1 | marginal gain | reading |
|---|---|---|---|
| smooth50 | .3037 | — | reference floor |
| Baseline+ | .3259 | +.0222 [+.0058, +.0403] vs smooth50 | about level to slightly ahead |
| D gate | .3987 | +.0728 [+.0649, +.0808] vs Baseline+ | the stoppage detector |
| ship (gate + in-gap) | .4039 | +.0052 [+.0038, +.0065] vs gate | the decoder's original credit |
| **frozen master** | **.4247** | **+.0208 [+.0169, +.0250] vs ship** | **0/10 games negative** |

| comparison | delta | 95% game-cluster CI | games negative |
|---|---|---|---|
| master − ship | **+.0208** | [+.0169, +.0250] | 0/10 |
| master − Baseline+ | +.0988 | [+.0909, +.1068] | 0/10 |

Against the organizers' reference, the gate-plus-decoder stack is +.1003 [+.0877, +.1130]. The final build and that reference were never measured as a pair, so we do not quote the arithmetic difference.

The gain is smaller on our match. We scale each rule by the share of our match it can touch. On that basis the final build should beat the gate-plus-decoder stack by +.011 to +.016, not the +.0208 [+.0169, +.0250] measured on the practice set. The practice set is about .06 easier, and the newer rules reach less of our match. Everything that moved moved for one reason: the D layer. The gate supplies most of the +.0988 [+.0909, +.1068] over Baseline+. The grown, sub-gap and floor rules add a few thousandths each.

What the file itself looks like, counted row by row from the frozen CSV:

| diagnostic | frozen CSV |
|---|---|
| rows / graded frames | 199,319 / 158,857 |
| H / A / D / X | 50,478 / 35,720 / 72,659 / 40,462 |
| D share of graded | .4574 |
| spells (maximal H/A runs) | 370 |
| D stretches | 138 (111 at least 1.5 s) |
| ball in play | 57.47 min of 105.9 min graded |
| sha256 | `ee7b5b6b…46d8` |

370 spells, 57.47 min of play and .4574 dead are in the right football neighborhood; they do **not** say the labels are right. Our match is dead-heavier than the football prior because of the cooling break and tracker-lost visible balls. These are label-free descriptions, not scores.

The public window, one run, reported and never a selector:

| run | spellF1 | macroF1 | predicted / GT spells |
|---|---|---|---|
| **ours (frozen)** | .4686 | .7199 | 18/17 |
| smooth125 | .4242 | .5812 | 16/17 |
| smooth50 (published floor) | .3256 | .6207 | 26/17 |
| smooth25 | .2868 | .6200 | 36/17 |
| raw | .0378 | .5972 | 131/17 |

Sorted by spellF1 we are first on both metrics, but seventeen true possessions is inside noise and the prediction has eighteen. Nothing was chosen here. Do not compare this table with the practice-set numbers above: different matches, different difficulty, different base.

There is no defensible absolute full-match score. Every interval above is a paired difference, the practice set is easier than the match, and the public window is only a direction check.

## 9. Limits, and what would fix them

The practice set has eight blind spots, and they bound what carries over to our match. Read the right-hand column as the effect on the numbers:

| blind spot | what it does to the numbers |
|---|---|
| no cooling breaks or video-review stops in the practice set | part of the practice-to-match gap is structural; our D share is legitimately higher |
| Metrica over-calls D when play continues after a foul (advantage) | 3 of 10 games; 20× spread in its own present-D share |
| short gaps (2–4 s) | ~10 per game, 0.5% of graded; in-gap and sub-gap gains are probably a little high |
| resting balls we still see in the practice set | .19 / .085 of them, against .45 in clean data; so grown D is measured pessimistically |
| how exactly a gap starts and ends | we jitter gap edges by about 1 s s (standard deviation); the spread we fitted to our match is 2×–4× larger, so the practice set is too tidy here |
| geometry (checked, closed) | nearest player 2.29 m vs 2.35 m; the nearest-player cue is not inflated |
| ten games, three seeds, one severity | the intervals ignore that our degrader may be wrong, and we have one match |
| level, not ranking | .3037 on the practice set against .243 published; only orderings transfer |

After the table, the biggest weakness is recall, not D. On the practice set 79% of real possessions are missed, so about one in five is caught. That figure is from the gate-plus-decoder stack on practice data, not a match measurement, which would need labels we do not have. Hand-overs are most of the loss, and much of it has no tracked ball.

Hand-over scatter is **evidence-limited, not rule-limited**. 46.9% [43.9%, 49.5%] of matched hand-overs are over a second off (31.2% early, 15.7% late), and 93.4% of that tail is the 2.5 m nearest-player cue itself. A duel has no timing information, and on a pass the receiver is nearest long before the touch. Only 0.69% of tail flips are moved by half a second or more by any stage after Baseline+. So no rule applied after the fact comes near the +.0796 that perfect hand-over timing would add (the **oracle**). The way forward is a detector of the moment a player touches the ball, accurate to about 0.16 s. On this tracking the ball is smoothed over several frames, which puts that out of reach.

A different data source would help in three ways. Tracking that records each touch would attack the hand-over tail and reopen the shelved idea of anchoring hand-overs to detected touches. Practice matches with drinks breaks and video-review stops would test the learned methods where they gain, since their advantage is confined to dead-frame motion that the ten games lack. A GPU ball detector would add ball coverage but not touch timing, and nothing may depend on it, since we assume no graphics card.

We would not retry rebuilding touches from ball motion, explicit-duration priors, a learned decoder in place of the gate, splitting long D blocks, or guessing the restarting team from the type of stoppage. Each has a decisive negative above.

## What changed since the freeze

The freeze itself did not move: the submission is still the same file, and the headline result is still +.0208 [+.0169, +.0250] against the gate-plus-decoder stack. What changed is how much of it we are willing to claim. Post-freeze studies re-measured the small components, asked whether the practice ranking survives a different draw or a different severity level (how much ball the degrader drops), took the two large orderings to twenty held-out SkillCorner games, and corrected how we measured the live share of the match clock. None of them reopened the frozen file; all of them changed what a reader should believe about the numbers in it.

The in-gap decoder is the clearest example. It was credited +.0052 [+.0038, +.0065] when it landed on the gate-only stack; on the current stack its leave-one-out marginal is +.0017 [+.0004, +.0030]. Those are different objects: most of the fall is work that grown D and the short-spell floor now do first, not a reversal of the decoder. The small number is also not sign-established. Any practice-set difference below .004 is quoted as *level with zero*, and this one is well below it. Six degradation draws span −.0009 to +.0031. The two-level interval, which resamples draws as well as games, is [−.0011, +.0041] and includes zero. We keep the stage, since removing it is not justified either, and stop calling it a gain.

The big practice-set result is not a lucky draw or a lucky severity. Fresh degradation draws give +.0203 [+.0166, +.0241] over six draws, with the three-seed number from the results section inside that interval. Scaling the practice data to 0.8× and 1.2× of its fitted severity leaves the same order, and the smallest adjacent margin is +.0195 at the cleaner level. These are ranking checks, not a second practice set, and no level was re-selected.

On twenty held-out SkillCorner games the two large orderings keep their sign under all four rules we use to derive D labels there: the final build beats Baseline+ by +.0361 to +.0567, and the gate-plus-decoder stack beats it by +.0215 to +.0407. The third ordering, the final build against the gate-plus-decoder stack, is positive but only +.0092 to +.0166, below the 0.02 bar we require for a held-out claim. The held-out set therefore confirms the two large gains and cannot resolve the small one; the practice-set grid still decides it.

The ball-in-play check had the wrong clock in its ratio, now fixed. The final file has 57.47 min of live play on a 103.95 min playing clock (the length of the two halves), a live share of 55.28%; the ten external games average 58.21% ± 6.38%. On the rate axis that is −0.46 standard deviations from the external mean — inside the noise, not the shortfall a raw-minute comparison suggested. The raw-minute axis is confounded by our unusually long clock and is no longer quoted.

The stage attribution is a substitution result, not a sum. Removing one stage at a time, with one degradation draw, puts the floor at +.0114, sub-gap at +.0062, grown D at +.0045 and in-gap at −.0009; the three post-freeze stages sum to about +.0221 against the measured +.0208 [+.0169, +.0250], an interaction of −.0013. Most of that interaction is grown D handing the floor work it used to do. Read the table as what each stage is worth given the other four, not as a budget that adds up.

## Code, data and acknowledgements

The build is one command in `solution/`: `python -m possession.make_submission`. The pipeline modules are `baseline_plus.py`, `dgate.py`, `growd.py`, `ingap.py`, `subgap.py` and `postprocess.py`; the harness is `dev_v1.py`; the renderer is `render.py`. This page is built by `scripts/walkthrough_numbers.py` and `scripts/build_walkthrough.py` from `numbers.json`.

The practice data is public tracking from DFL, Metrica and SkillCorner, used under their licenses and never redistributed. The match data is the organizer's and stays on this machine. The scoring code is the organizer's `evaluation/possession_metrics.py`, imported, not re-implemented. Figures that render tracking are local-only under D-002; the public profile replaces them with placeholders.

## Glossary

Every technical term used on this page, in plain words. Listed alphabetically; each entry gives the football meaning first, then our name for it.

| term | plain meaning |
|---|---|
| **AUC** | Area under the ROC curve: how well a score separates two classes; half is chance, one is perfect. |
| **ball-free model** | A classifier that estimates who has the ball from the players, without the ball. |
| **ball-missing run (gap)** | A stretch with no tracked ball. It can be a stoppage or a tracker dropout. |
| **Baseline+** | The first stage: nearest player within a radius, otherwise the last owner, with a merge. |
| **CI (confidence interval)** | The range a number could plausibly take. Narrow means well measured. |
| **classifier** | A model that maps frames to a probability, here of the home team having the ball. |
| **CSV** | The submission file: one row per frame with the label. |
| **D gate** | The rule that turns a long ball-missing stretch with still players into D. |
| **degradation** | The process that makes clean practice tracking look like ours: drop the ball, smooth it, jitter the edges. |
| **degradation draw (seed)** | One random version of the degradation; the seeds are separate draws. |
| **`dev-v1` (practice set)** | Our internal name for the ten public matches with answers, degraded to look like our tracking. This page says *practice set*. |
| **DFL** | The German football league's tracking data, one of the practice-set providers. |
| **eligible frames** | Frames inside a gap the gate left live, which the decoder may rewrite. |
| **emission** | A per-frame score from the ball-free model: how likely each team is to have the ball. |
| **false positive / false negative** | A predicted spell with no true match / a true spell we miss. |
| **final build (frozen)** | The stack that produced the submission file, fixed before this page was written. |
| **flicker** | A very short spurious spell, usually one frame. |
| **game-cluster interval** | A confidence interval from resampling the ten practice games, not the frames. |
| **gate-only stack** | The gate on top of Baseline+, before the in-gap decoder, grown D, the sub-gap rule and the floor. |
| **gate-plus-decoder stack (ship)** | The build with the gate and the in-gap decoder only. The final build adds grown D, the sub-gap rule and the floor. |
| **graded frames** | The frames inside the two halves that are scored; the rest are X. |
| **grown D** | A rule that extends a visible resting ball to the stoppage around it. |
| **H / A / D / X** | The four labels: Barcelona, Athletic, dead ball (a stoppage), and not graded. A frame with no grade is X. |
| **hand-over** | The ball changes team. |
| **HMM (hidden Markov model)** | A sequence model with hidden states (here H/A) and a cost for switching. |
| **HSMM** | A hidden Markov model with an explicit duration model; a variant we tried and dropped. |
| **hysteresis** | A rule that changes the owner only when the evidence is clear, so one noisy frame cannot flip it. |
| **Hz** | Times per second. Our labels come once per frame, at the frame rate. |
| **in-gap decoder** | The stage that decides H or A inside a gap the gate left live, from the players' shape. |
| **IoU (intersection over union)** | Shared time divided by the time either spell covers. One means identical. |
| **level with zero** | Our rule: a practice-set delta below .004 has no established sign. |
| **macroF1** | Per-frame F-score averaged over H, A and D; a tie-break, not the ranking score. |
| **marginal (leave-one-out)** | The score change from removing one stage and re-running the whole build. |
| **MBR** | Minimum Bayes risk decoding: pick the label sequence with the best expected score. We tried it and dropped it. |
| **merge** | Joining a very short spell into the longer ones around it. |
| **Metrica / SkillCorner** | Two more providers of public match tracking data. |
| **oracle** | A score with perfect knowledge of the answer: an upper bound no real rule reaches. |
| **owner** | The team that last touched the ball and still controls it. |
| **P(H)** | The ball-free probability that Barcelona has the ball on a frame. |
| **playing clock** | The length of the two halves on the match clock, stoppages included. |
| **possession (spell)** | A run of frames in which one team has the ball. A dead ball ends it; the score counts these runs, not frames. |
| **precision / recall** | Of the spells we predict, how many are right. Of the true spells, how many we find. |
| **primary cell** | One of the six practice-set configurations: three ball-loss severities times two thresholds for how still the players must be. The headline averages them. |
| **public window** | The five public minutes with answers in our match; report-only, never used to choose. |
| **report-only** | A number we report but never use to choose a method or a threshold. |
| **resting ball** | A ball lying still, for example before a free kick. |
| **severity (level)** | How much ball the degrader drops. The practice set uses the severity fitted to our match. |
| **short-spell floor** | The rule that deletes H/A spells shorter than a second and a half. |
| **`smooth50`** | The organizers' reference: nearest player, majority-smoothed over two seconds. |
| **spellF1** | The ranking score (spell-level F-score): match predicted possessions to true ones one-to-one by IoU, then average the F-score over five thresholds. |
| **stoppage / dead ball** | Play is stopped by the referee: a foul, throw-in, goal kick, injury or break. The ball can still be on the pitch. |
| **sub-gap rule** | A second, weaker gate that finds a stoppage inside a gap the main gate left alone. |
| **substitutes / interaction** | Stage gains overlap: removing two stages does not add their separate gains. |
| **switching penalty** | A cost added whenever the model changes team; it stops flicker. |
| **tracker / tracking** | The system that follows players and ball through the match footage. It sometimes loses the ball. |
| **tracker dropout** | The tracking loses the ball while play continues. |
| **Viterbi** | The algorithm that finds the most likely state sequence under the switching cost. |
| **Viterbi decode** | Pick the single most likely label sequence. We tried it in place of our rules and dropped it. |


## Appendix: traceability

Every measured number in the prose is a key in `notebooks/numbers.json`, with its value, source, exact command and whether `--check` recomputes it from the live pipeline. The table lists the headline keys; the rest are in that file, and the figure table names every figure.

### The process, stage by stage

The full diagram for a reader who wants the process: each stage's decision, and the measured marginal on each arrow. The notes under the figure say why the arrows are substitutes and not a sum.

<img src="figs/fig_pipeline_flow.png" style="width:100%">
<div class="caption" style="font-size:0.9em; color:#555; margin:2px 0 14px 0;">The pipeline left to right: two jobs (who owns the ball, when it is dead) and a clean-up. Each arrow label is the score lost when that stage is removed from the full build (spellF1, practice set); the labels do not add up to the total.</div>

### The gain ladder

The supporting picture for the results section: Panel A is the practice-set selection and Panel B the public five minutes. The notes under the figure carry the substitution caveat and the .004 rule.

<img src="figs/fig_gain_waterfall.png" style="width:100%">
<div class="caption" style="font-size:0.9em; color:#555; margin:2px 0 14px 0;">Panel A: the practice-set ladder, with each stage's leave-one-out marginal and the .004 line below which a practice-set difference has no established sign. Panel B: the public five minutes, report only.</div>

| key | value | section | recomputed |
|---|---|---|---|
| master_dev | .4247 | 7 | no |
| master_over_ship | +.0208 [+.0169, +.0250] | 7, post | no |
| master_over_baseline_plus | +.0988 [+.0909, +.1068] | start, 7 | no |
| master_neg | 0/10 | 7 | no |
| expected_master | +.011 to +.016 | 7 | no |
| smooth50_dev | .3037 | 3, 4 | yes |
| baseline_plus_dev | .3259 | 4 | yes |
| gate_dev | .3987 | 4 | yes |
| ship_dev | .4039 | 4 | yes |
| ship_over_smooth50 | +.1003 [+.0877, +.1130] | start, 7 | yes |
| gate_over_baseline_plus | +.0728 [+.0649, +.0808] | start, 4 | yes |
| ingap_over_gate | +.0052 [+.0038, +.0065] | ingap, 4, post | yes |
| grown_delta | +.0055 | 4 | no |
| subgap_delta | +.0044 | 4 | no |
| postprocess_delta | +.0099 | 4 | no |
| postprocess_delta_current | +.0113 | 4 | no |
| attrib_floor | +.0114 | post | no |
| attrib_subgap | +.0062 | post | no |
| attrib_grown | +.0045 | post | no |
| ingap_marginal | +.0017 | start, ingap, post | no |
| sign_threshold | .004 | start, ingap, post, glossary, appendix | no |
| floor_real_short | 84 | — | no |
| floor_short_total | 464 | — | no |
| floor_false_short | 380 | — | no |
| explain_clean | 1.00 | — | yes |
| explain_flicker | .74 | — | yes |
| explain_missed | .80 | — | yes |
| ingap_added_spells | 2,289 | ingap | no |
| ingap_removed_short | 921 | ingap | no |
| ingap_flicker_before | 6.25% | ingap | no |
| ingap_flicker_after | 4.18% | ingap | no |
| ingap_neg_games | 3/10 | ingap | no |
| ingap_cells_pos | 6/6 | ingap | no |
| demo_ingap_ph | .8 | ingap | no |
| demo_split_iou | .33 | ingap | yes |
| demo_continuous_iou | 1.00 | ingap | yes |
| pred_spells | 370 | 7 | yes |
| d_stretches | 138 | 7 | yes |
| pred_d_share_graded | .4574 | 7 | yes |
| ball_in_play_min | 57.47 min | 7, post | yes |
| graded_min | 105.9 min | 7 | yes |
| csv_sha8 | ee7b5b6b…46d8 | 4 | yes |
| public_ship_spellf1 | .4686 | 7 | no |
| public_ship_macro | .7199 | 7 | no |
| public_ship_spells | 18 | 7 | no |
| missed_share | 79% | 8 | no |
| handover_tail | 46.9% | 8 | no |
| handover_cue_share | 93.4% | 8 | no |
| stage_moved | 0.69% | 8 | no |
| handover_oracle | +.0796 | 8 | no |
| e25_dead_share | 85.6% | 5 | no |
| e36_dead_share | 84.5% | 5 | no |
| shape_width_diff_m | 2.5 m | 5 | yes |
| shape_dead_gap_m | −0.4 m | 5 | yes |

| figure | section | profile |
|---|---|---|
| fig_gain_waterfall | appendix | both |
| fig_gap_ambiguity | ingap | both |
| fig_gate_instruments | 5 | both |
| fig_ingap_before_after | ingap | both |
| fig_ingap_locator | ingap | both |
| fig_ingap_result | ingap | both |
| fig_ingap_steps | ingap | both |
| fig_killed | 6 | both |
| fig_match_gap | 1 | local only |
| fig_missingness | 3 | both |
| fig_nearest_player | 1 | both |
| fig_pipeline_flow | appendix | both |
| fig_practice_clean | 3 | local only |
| fig_practice_degraded | 3 | local only |
| fig_shape_deform | 5 | local only |
| fig_spellf1 | 2 | both |
| fig_spellf1_curve | 2 | both |
| fig_spellf1_explainer | 2 | both |
| fig_stage_baseline_plus | 4 | local only |
| fig_stage_gate | 4 | local only |
| fig_stage_grown | 4 | local only |
| fig_stage_ingap | 4 | local only |
| fig_stage_smooth50 | 4 | local only |
| fig_stage_subgap | 4 | local only |
| fig_three_labels | 1 | both |
| fig_width_schematic | 5 | both |